In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/turbine_hourly.csv", parse_dates=["ts_hour"])
df = df.sort_values(["turbine_id", "ts_hour"])

# capacity factor
rated_power_kw = 3000
df["capacity_factor"] = df["avg_power_kw"] / rated_power_kw

# daily energy per turbine
daily = (
    df.groupby(["turbine_id", pd.Grouper(key="ts_hour", freq="D")])["energy_kwh"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(12, 4))
for t in df["turbine_id"].unique():
    subset = daily[daily["turbine_id"] == t]
    plt.plot(subset["ts_hour"], subset["energy_kwh"], label=t, alpha=0.7)
plt.legend()
plt.title("Daily Energy Yield per Turbine")
plt.ylabel("Energy (kWh)")
plt.tight_layout()
plt.show()

# monthly capacity factor
monthly_cf = (
    df.set_index("ts_hour")
      .groupby("turbine_id")["capacity_factor"]
      .resample("M")
      .mean()
      .reset_index()
)

plt.figure(figsize=(10, 4))
sns.lineplot(data=monthly_cf, x="ts_hour", y="capacity_factor", hue="turbine_id")
plt.title("Monthly Capacity Factor")
plt.ylabel("Capacity factor")
plt.tight_layout()
plt.show()
